## OOP(oBJECT ORIENTED PROGRAMMING)

In [2]:
class Person:
    pass

The moment Python executes this, it creates a class object — an instance of the built-in type class. Yes, classes are themselves objects:

In [3]:
print(type(Person))

<class 'type'>


type is the "class of classes" — the mechanism that builds every class you ever write. This is deep enough that you don't need to fully absorb it now, just know it exists — it's why frameworks can do wild things like generate classes dynamically at runtime.

In [4]:
#the constructor __init__

class Person:
    def __init__(self,name,age):
        self.name = name
        self.age = age

 __init__ isn't technically "the constructor" in the strictest sense (that's __new__, covered below) — it's the initializer, called automatically right after the object is created, to set up its starting state.

In [5]:
p = Person("Siva",25)

What actually happens on this line, in order:

Python calls Person.__new__(Person) → creates a blank, empty object in memory
Python calls Person.__init__(that_object, "Siva", 25) → fills in .name and .age on it
The fully-built object gets bound to p

In [6]:
#3. self — deep understanding, not just "it's required"
#self is not a keyword — it's just a convention. 
# You could name it anything:
class Person:
    def __init__(banana,name):
        banana.name = name



In [10]:
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1     # modifies THIS instance's count, not a shared one

c1 = Counter()
c2 = Counter()
c1.increment()
c1.increment()
print(c1.count, c2.count)     # 2 0 — separate state per instance

2 0


In [ ]:
#Instance attributes vs class attributes — 
# the trap everyone falls into
class Dog:
    tricks = []

    def __init__(self,name):
        self.name = name
  # instance attribute — safe, unique per object
    def add_tricks(self,trick):
        self.tricks.append(trick)
 # modifies the SHARED class-level list!
d1 = Dog("Rex")
d2 = Dog("Fido")
d1.add_tricks("sit")
print(d2.tricks)
 # ['sit']  ← leaked into d2! Not what you'd expect

['sit']


This is one of the most common real bugs in Python OOP. Mutable class attributes (lists, dicts) are shared across every instance unless you explicitly create them per-instance in __init__:

In [23]:
#Reading vs writing attributes — where lookup actually happens
class Dog:
    species = "Canine"

d = Dog()
print(d.species)
d.species = "Wolf"
print(d.species)
print(Dog.species)

Canine
Wolf
Canine


Attribute lookup order: instance __dict__ first, then class __dict__, then parent classes (this chains into MRO, coming later). Writing an attribute always writes to the instance, never silently mutates the class — unless you explicitly do Dog.species = ....

In [24]:
class Circle:
    pi = 3.14159

    def __init__(self, radius):
        self.radius = radius

    def area(self):                          # instance method — needs self
        return Circle.pi * self.radius ** 2

    @classmethod
    def unit_circle(cls):                     # classmethod — needs cls, not self
        return cls(radius=1)                    # cls(...) == Circle(...), but works for subclasses too

    @staticmethod
    def is_valid_radius(radius):                # staticmethod — needs neither
        return radius > 0

__new__ is responsible for actually creating and returning the object (rarely overridden — mostly relevant for immutable types like subclassing str/tuple, or singleton patterns). __init__ just configures an already-created object and returns nothing (None implicitly). 99% of the time you'll only ever touch __init__ — but knowing __new__ exists explains why __init__ never needs a return statement.

In [25]:
#__new__ vs __init__ — the real object lifecycle
class Person:
    def __new__(cls, *args, **kwargs):
        print("Creating the object")
        instance = super().__new__(cls)
        return instance

    def __init__(self, name):
        print("Initializing the object")
        self.name = name

p = Person("Siva")
# Output:
# Creating the object
# Initializing the object

Creating the object
Initializing the object


In [26]:
#__repr__ vs __str__ — precise difference
class Person:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Person(name={self.name!r})"    # unambiguous, for developers/debugging

    def __str__(self):
        return f"{self.name}"                      # readable, for end users

p = Person("Siva")
print(p)          # uses __str__ → "Siva"
print([p])        # uses __repr__ inside containers → [Person(name='Siva')]
repr(p)             # explicit → "Person(name='Siva')"

Siva
[Person(name='Siva')]


"Person(name='Siva')"

In [27]:
#9. Checking types and identity
p = Person("Siva")

print(isinstance(p, Person))       # True — is p an instance of Person (or subclass)?
print(type(p) == Person)            # True, but isinstance() is preferred — respects inheritance
print(p.__class__)                    # Person — the class this object was made from

True
True
<class '__main__.Person'>


In [28]:
#Python's version — but split into three, not two

class Person:
    count = 0                              # class attribute (like Java's static field)

    def instance_method(self):               # like Java's instance method
        return f"needs an object, self={self}"

    @classmethod
    def class_method(cls):                    # NO Java equivalent — unique to Python
        return f"needs the class, cls={cls}"

    @staticmethod
    def static_method():                       # equivalent to Java's static method
        return "needs neither"

Inheritance + super() — the mechanism behind class MyNode(BaseNode)

In [ ]:
#Basic inheritance — reminder
